# Load necessary dependancies

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch import optim, nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Device Agnostic

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

# Load index files

In [ ]:
base_path = './drive/MyDrive/mtcaic3'
train_df = pd.read_csv(os.path.join(base_path, 'train.csv'))
validation_df = pd.read_csv(os.path.join(base_path, 'validation.csv'))
test_df = pd.read_csv(os.path.join(base_path, 'test.csv'))

label_map = {"Right":0, "Left":1, "Backward":2, "Forward":3}
reverse_label_map = {0: "Right", 1:"Left", 2:"Backward", 3:"Forward"}

# Preparations

## dataset.py

In [ ]:
class BCICustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
            return self.features[idx], self.labels[idx]

## dataloader.py

In [ ]:
def load_trial_data(row, base_path='.'):
    # Determine dataset type based on ID range
    id_num = row['id']
    if id_num <= 4800:
        dataset = 'train'
    elif id_num <= 4900:
        dataset = 'validation'
    else:
        dataset = 'test'

    # Construct the path to EEGdata.csv
    eeg_path = f"{base_path}/{row['task']}/{dataset}/{row['subject_id']}/{row['trial_session']}/EEGdata.csv"

    # Load the entire EEG file
    eeg_data = pd.read_csv(eeg_path)

    # Calculate indices for the specific trial
    trial_num = int(row['trial'])
    if row['task'] == 'MI':
        samples_per_trial = 2250  # 9 seconds * 250 Hz
    else:  # SSVEP
        samples_per_trial = 1750  # 7 seconds * 250 Hz

    start_idx = (trial_num - 1) * samples_per_trial
    end_idx = start_idx + samples_per_trial - 1

    # Extract the trial data
    trial_data = eeg_data.iloc[start_idx:end_idx+1]
    return trial_data

def data_loader(path_map_df, base_path, shuffle=True, batch_size=32):
    trials = []
    labels = []
    for i in range(len(path_map_df)):
        trial_data = load_trial_data(base_path=base_path, row=path_map_df.iloc[i, :])
        if trial_data.isna().any(axis=None):
            print(f"Trial #{i} is corrupted, skiping...")
            continue
        else:
            data = trial_data.drop(["Time", "AccX", "AccY", "AccZ", "Gyro1", "Gyro2", "Gyro3", "Battery", "Counter", "Validation"], axis=1).T
            trials.append(data.values)
            labels.append(label_map[path_map_df.iloc[i, -1]])
    dataset = BCICustomDataset(features=torch.tensor(np.array(trials), dtype=torch.float32), labels=torch.tensor(labels, dtype=torch.long))
    dataloader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=shuffle)
    return dataloader

## metrics.py

In [ ]:
try:
    from torchmetrics import Accuracy, F1Score, Precision, Recall
except:
    print("Can't find torchmetrics, installing...")
    !pip install torchmetrics -q
    from torchmetrics import Accuracy, F1Score, Precision, Recall

def set_metrics(num_classes, task="multiclass", device="cpu"):
    accuracy_score = Accuracy(task=task, num_classes=num_classes).to(device)
    f1_score = F1Score(task=task, num_classes=num_classes).to(device)
    precision = Precision(task=task, num_classes=num_classes).to(device)
    recall = Recall(task=task, num_classes=num_classes).to(device)
    return accuracy_score, f1_score, precision, recall

def classification_report(num_classes, y_true, y_pred, task="multiclass", device="cpu"):
    accuracy_score, f1_score, precision, recall = set_metrics(num_classes, task="multiclass", device="cpu")
    report = {
        "Accuracy":     accuracy_score(y_pred, y_true).item(),
        "Precision":    precision(y_pred, y_true).item(),
        "Recall":       recall(y_pred, y_true).item(),
        "F1-Score":     f1_score(y_pred, y_true).item()
        }
    return report

## utils.py

In [ ]:
import numpy as np
import random
import torch
from pathlib import Path
def set_seed(seed:int=42):
    """
    sets the seed in all needed libraries (torch, torch.cuda, numpy, and random for general python code)

    Arguments:
    seed:int=42, the seed
    """
    np.random.seed(seed) # Sets NumPy random seed
    torch.manual_seed(seed) # Sets PyTorch's random seed
    torch.cuda.manual_seed(seed) # Sets PyTorch's random seed on CUDA
    torch.cuda.manual_seed_all(seed) # Sets PyTorch's random seed on all objects on the GPU
    random.seed(seed) # Sets the random seed on all python objects

def save_model(model, model_name, device):
    try:
        scripted_model = torch.jit.script(model)  # safer long term
    except Exception as e:
        print(f"[WARN] Scripting failed, falling back to tracing: {e}")
        example_input = torch.randn(1, 1, 28, 28).to(device)
        scripted_model = torch.jit.trace(model, example_input)
    base_dir = Path("models/")
    base_dir.mkdir(parents=True, exist_ok=True)
    scripted_model.save(f"models/{model_name}.pt")

## preproc.py

In [ ]:
def seperate_task(task, train_df=train_df, validation_df=validation_df, test_df=test_df):
    try:
        task_df_train = train_df.loc[train_df["task"] == task]
        task_df_val = validation_df.loc[validation_df["task"] == task]
        task_df_test = test_df.loc[test_df["task"] == task]
        return task_df_train, task_df_val, task_df_test
    except:
        return f"The task {task+1} does exist, please choose either SSVEP or MI"

# Prepare DataLoaders

## SSVEP Task

In [ ]:
# Load the train, validation, and test data for the SSVEP Task
ssvep = seperate_task("SSVEP")
train_ssvep, val_ssvep, test_ssvep = ssvep

# Preare the DataLoaders
batch_size = 32

print("Preparing 'train_dataloader_ssvep'...")
train_dataloader_ssvep = data_loader(train_ssvep, base_path=base_path, shuffle=True, batch_size=batch_size)
print()

print("Preparing 'val_dataloader_ssvep'...")
val_dataloader_ssvep = data_loader(val_ssvep, base_path=base_path, shuffle=False, batch_size=batch_size)
print()

# print("Preparing 'test_dataloader_ssvep'...")
# test_dataloader_ssvep = data_loader(test_ssvep, base_path=base_path, shuffle=False)
# print()

print("SSVEP DataLoaders are Ready!")

Preparing 'train_dataloader_ssvep'...

Preparing 'val_dataloader_ssvep'...

SSVEP DataLoaders are Ready!


## MI Task

In [ ]:
# Load the train, validation, and test data for the MI Task
mi = seperate_task("MI")
train_mi, val_mi, test_mi = mi

# Prepare the DataLoaders
print("Preparing 'train_dataloader_mi'...")
train_dataloader_mi = data_loader(train_mi, base_path=base_path, shuffle=True, batch_size=batch_size)
print()

print("Preparing 'val_dataloader_mi'...")
val_dataloader_mi = data_loader(val_mi, base_path=base_path, shuffle=False, batch_size=batch_size)
print()

# print("Preparing 'test_dataloader_mi'...")
# test_dataloader_mi = data_loader(test_mi, base_path=base_path, shuffle=False)
# print()

print("MI DataLoaders are Ready!")

Preparing 'train_dataloader_mi'...
Trial #889 is corrupted, skiping...

Preparing 'val_dataloader_mi'...

MI DataLoaders are Ready!


# Sample observation

In [ ]:
train_ssvep_features, train_ssvep_label = next(iter(train_dataloader_ssvep))
train_mi_features, train_mi_label = next(iter(train_dataloader_mi))

# model.py

In [ ]:
class EEGNet(nn.Module):
    def __init__(self, fs, in_features, num_channels, num_classes, f1=8, D=2):
        super().__init__()
        f2 = f1*D
        self.num_channels = num_channels
        self.fs = fs
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=f1, kernel_size=(1, fs//2), bias=False),
            nn.BatchNorm2d(f1),
            nn.Conv2d(in_channels=f1, out_channels=f2, kernel_size=(num_channels, 1), groups=f1, bias=False),
            nn.BatchNorm2d(f2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1,4)),
            nn.Dropout(p=0.5)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=f2, out_channels=f2, kernel_size=(1, 16), groups=f2, bias=False),
            nn.BatchNorm2d(f2),
            nn.Conv2d(in_channels=f2, out_channels=f2, kernel_size=1, bias=False),
            nn.BatchNorm2d(f2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1,8)),
            nn.Dropout(p=0.5),
            nn.Flatten()
        )

        self.fc = nn.Sequential(
            nn.Linear(in_features=in_features, out_features=num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.fc(x)
        return x

# Instantiate the model

In [ ]:
fs, num_channels, num_classes, f1, D = 250, 8, 4, 8, 2
in_features = 768
ssvep_model_0 = EEGNet(
    in_features=in_features,
    fs=fs,
    num_channels=num_channels,
    num_classes=num_classes,
    f1=f1,
    D=D
).to(device)

loss_fn = nn.CrossEntropyLoss(reduction="mean")

lr = 3e-5
optimizer = optim.AdamW(params=ssvep_model_0.parameters(), lr=lr, weight_decay=1e-3)

# Training the model

In [ ]:
def train_test_loop(epochs, model, loss_fn, optimizer, train_dataloader, test_dataloader, device="cpu", verbose=False):
    accuracy_score, f1_score, precision, recall = set_metrics(num_classes=9, task="multiclass", device=device)
    for epoch in tqdm(range(epochs)):
        print(f"Epoch #{epoch}")
        train_loss = 0
        test_loss = 0
        train_acc = 0
        test_acc = 0
        model.train()
        for batch, (X_train, y_train) in enumerate(train_dataloader):
            X_train, y_train = X_train.unsqueeze(1).to(device), y_train.squeeze().to(device)
            y_pred = model(X_train).squeeze()
            loss = loss_fn(y_pred, y_train)
            train_loss += loss.item()
            accuracy_score.update(torch.softmax(y_pred, dim=1).argmax(dim=1), y_train)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if verbose and batch > 0 and batch%15==0:
                print(f"[INFO] Batch #{batch}: Average Train Loss: {(train_loss/(batch+1)):.3f}")
        train_acc = accuracy_score.compute()
        accuracy_score.reset()

        model.eval()
        with torch.inference_mode():
            for X_test, y_test in test_dataloader:
                X_test, y_test = X_test.unsqueeze(1).to(device), y_test.to(device)

                test_pred = model(X_test).squeeze()
                loss = loss_fn(test_pred, y_test.squeeze())
                test_loss += loss.item()
                accuracy_score.update(torch.softmax(test_pred, dim=1).argmax(dim=1), y_test)


        average_train_loss = train_loss/len(train_dataloader)

        average_test_loss = test_loss/len(test_dataloader)
        test_acc = accuracy_score.compute()
        accuracy_score.reset()

        print(f"[INFO] Average Train Loss: {average_train_loss:.3f}, Train Accuracy: {train_acc:.2%}, Average Test Loss: {average_test_loss:.3f}, Test Accuracy: {test_acc:.2%}")
    return model

In [ ]:
epochs = 10
model = train_test_loop(epochs, ssvep_model_0, loss_fn, optimizer, train_dataloader=train_dataloader_ssvep, test_dataloader=val_dataloader_ssvep, device=device, verbose=True)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch #0
[INFO] Batch #15: Average Train Loss: 1.456
[INFO] Batch #30: Average Train Loss: 1.443
[INFO] Batch #45: Average Train Loss: 1.440
[INFO] Batch #60: Average Train Loss: 1.443
[INFO] Average Train Loss: 1.440, Train Accuracy: 24.46%, Average Test Loss: 1.416, Test Accuracy: 24.00%
Epoch #1
[INFO] Batch #15: Average Train Loss: 1.413
[INFO] Batch #30: Average Train Loss: 1.421
[INFO] Batch #45: Average Train Loss: 1.430
[INFO] Batch #60: Average Train Loss: 1.435
[INFO] Average Train Loss: 1.437, Train Accuracy: 24.67%, Average Test Loss: 1.382, Test Accuracy: 32.00%
Epoch #2
[INFO] Batch #15: Average Train Loss: 1.428
[INFO] Batch #30: Average Train Loss: 1.428
[INFO] Batch #45: Average Train Loss: 1.427
[INFO] Batch #60: Average Train Loss: 1.425
[INFO] Average Train Loss: 1.426, Train Accuracy: 25.12%, Average Test Loss: 1.375, Test Accuracy: 32.00%
Epoch #3
[INFO] Batch #15: Average Train Loss: 1.405
[INFO] Batch #30: Average Train Loss: 1.412
[INFO] Batch #45: Average Trai